# Imports

In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.append('CISM')  # Replace with the actual path
from CISM.cism.graph.create_formatted_graph import GraphBuilder
import os
import pathlib

In [ ]:
DATA_DIR = '/groups/assafza_group/assafza/Datasets/CRC'

raw_csv_file = os.path.join(DATA_DIR, 'CRC_cells.csv')
raw_patient_class_file = os.path.join(DATA_DIR, 'CRC_patient_class.csv')
root_data_dir =  './'
output_graphs_dir = 'CRC_graphs/'

FANMOD_path = '/groups/assafza_group/assafza/FANMODPlus/Binary'
FANMOD_exe = 'LocalFANMOD'
output_dir = './FANMOD_output/'
cache_dir = './FANMOD_cache/'


In [ ]:
data = pd.read_csv(raw_csv_file)
relevant_columns = ["cellID", 'fov','centroid_x', 'centroid_y','class']
data = data[relevant_columns]
data.head()

In [ ]:
cell_types = data['class'].unique()


In [ ]:
data[relevant_columns].head()

In [ ]:
def extract_patient_id(fov):
    return int(fov.split('_')[0][1:])

def extract_fov_number(fov):
    return int(fov.split('_')[2][3:])
try:
    data['patient_id'] = data['fov'].apply(extract_patient_id)
    data['fov'] = data['fov'].apply(extract_fov_number)
    data.rename(columns={'cellID': 'cell_id','centroid_x':'centroid-0','centroid_y':'centroid-1'}, inplace=True)
except:
    pass
data = data[['patient_id','fov','cell_id','centroid-0','centroid-1','class']]
data.head()

# Build Graphs

In [ ]:
cell_mapper = {cell_type: cell_type for cell_type in cell_types} 
colnames_mapper_dict = {'cell_types' : 'class', #here we configure which column holds the information of cell type
                        'patient_id' : 'patient_id'}

g = GraphBuilder(data,cell_mapper, colnames_mapper_dict)

g.build_graph(path_to_output_dir=root_data_dir + output_graphs_dir, 
              #50 µm - the maximum distance between cells to be considered neighbours. 
              #The paramter value is 100 since the resolution is 0.5 µm.
              max_distance=125,
              exclude_cell_type= None)

cell_types = {v:k for k, v in g.common_cell_type_mapper.items()}


In [ ]:
cell_types = {'Bcell': 0, 'CD3T': 1, 'CD4T': 2, 'CD8T': 3, 'DC': 4, 'Endothelial': 5, 'Fibroblast': 6, 'Lymphatic': 7, 'Macrophage': 8, 'Neuron': 9, 'Neutrophil': 10, 'Plasma': 11, 'SMV': 12, 'Stroma': 13, 'Treg': 14, 'Tumor': 15, 'Unidentified': 16}
cells_type = {value: key for key, value in cell_types.items()}

In [ ]:

assert pathlib.Path(FANMOD_path + "//" + FANMOD_exe).exists(), "FANMOD executable not found"

# Run CISM

In [ ]:
from CISM.cism.cism import CISM

In [ ]:
motif_size = 4
iterations = 1000
#root_data_dir = '.'
#output_graphs_dir = 'CRC_graphs'



cism = CISM(fanmod_exe=FANMOD_exe,
            fanmod_path=FANMOD_path,
            network_dataset_root_path=root_data_dir,
            fanmod_output_root_path=output_dir,
            fanmod_cache_root_path=cache_dir,
            motif_size=motif_size,
            iterations=iterations)




In [ ]:
# adding the dataset
cism.add_dataset(output_graphs_dir, 
                 'Disease', 
                 'CRC', 
                 force_run_fanmod=False, 
                 force_parse=False, 
                 n_jobs=12)

In [ ]:
import pickle as pkl
pkl.dump(cism, open('cism_CRC_4.pkl', 'wb')) 

In [ ]:
from CISM.cism.cism import TissueStateDiscriminativeMotifs
from CISM.cism.cism import DiscriminativeFeatureKey

## Discriminative Motfs size 

In [ ]:
patient_class = pd.read_csv(raw_patient_class_file)
patient_class = patient_class[['Patient', 'Group']].dropna()
patient_class['Patient'] = patient_class['Patient'].astype(int)
patient_class.to_csv(f'{raw_patient_class_file}_processed.csv', index=False)

def map_class(survival_rate):
    if int(survival_rate) == 1:
        return 'CLR'
    elif int(survival_rate) == 2:
        return 'DII'
    else:
        raise ValueError(f"Unknown survival rate: {survival_rate}")

classes = ['CLR', 'DII']


In [ ]:
#patient_class.to_csv('patient_class_simple.csv', index=False, header=False)

In [ ]:

discriminator = TissueStateDiscriminativeMotifs(cism=cism,
                                                tissue_state_csv_path='patient_class_simple.csv',
                                                tissue_state_to_string=None,
                                                common_cells_type=cell_types,
                                                tissue_state_func=map_class)

In [ ]:
discriminator.get_patients_class(classes=classes).groupby('patient_class').count()

In [ ]:
import warnings # To suppress some warnings
import seaborn as sns

# Suppress the specific FutureWarning
warnings.filterwarnings("ignore", category=FutureWarning, module="seaborn")

class_to_color = {
    'CLR': 'red',
    'DII': 'blue',
}

sns.set(font_scale=1)
sns.set_style("white")

discover_result = discriminator.discover(extract_by=DiscriminativeFeatureKey.STRUCTURE_AND_CELL_IDENTITIES, classes=classes)

discover_result.plot_number_of_motifs_versus_discrimination_stringency_parameter(class_to_color=class_to_color)

In [ ]:
from CISM.cism.cism import HardDiscriminativeFC, SoftDiscriminativeFC, DiscriminativeFeatureKey
import itertools

In [ ]:


results_table = None
random_state = np.random.RandomState(0)

for left_class, right_class in itertools.combinations(classes, 2):
    for th in [0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.7, 0.75, 0.8]:
        try:
            print(f'task: {left_class} - {right_class}')
            featureConf = HardDiscriminativeFC(
                extract_by=DiscriminativeFeatureKey.STRUCTURE_AND_CELL_IDENTITIES,
                use_cells_type_composition=False,
                use_motifs=True,
                shared_percentage=th,
                max_class_features=30,
                labels=[left_class, right_class])

            motif_space_features = (discriminator
                                    .get_features(feature_conf=featureConf,
                                                  exclude_patients=[],
                                                  n_jobs=1))

            optimal_result = (discriminator
                              .analyze_motifs(feature_conf=featureConf,
                                              exclude_patients=[],
                                              n_jobs=1,
                                              random_state=random_state))

            print(f'classes: {[left_class, right_class]} '
                  f'th:{th} score: {optimal_result.get_roc_auc_score()}')

            all_motif_features = []
            for idx, row in motif_space_features.iterrows():
                all_motif_features += row['features']
            all_motif_features = list(set(all_motif_features))

            motif_freq = []
            motif_count = []
            motifs_dataset = discriminator.cism.motifs_dataset
            for motif_id in all_motif_features:
                motif_mean_freq = motifs_dataset[motifs_dataset['ID'] == motif_id]['Freq'].mean()
                motif_mean_count = motifs_dataset[motifs_dataset['ID'] == motif_id]['Count'].mean()
                motif_patients_count = motifs_dataset[motifs_dataset['ID'] == motif_id]['Patient_uId'].nunique()
                motif_freq.append(motif_mean_freq)
                motif_count.append(motif_mean_count)

            results_table = pd.concat([pd.DataFrame(
                                            {'th': th,
                                             'task': left_class + '-' + right_class,
                                             'left_class': left_class,
                                             'right_class': right_class,
                                             'roc_auc_score': optimal_result.get_roc_auc_score(),
                                             'all_motif_features': all_motif_features,
                                             'motif_mean_freq': motif_freq,
                                             'motif_mean_count': motif_count,
                                             'patient_count': motif_patients_count}),
                                             results_table], ignore_index=True)
        except Exception as e:
            break

In [ ]:
a = results_table[['th','roc_auc_score']].groupby('th').mean()

In [ ]:
b = results_table[['th','roc_auc_score']].groupby('th').mean()

In [ ]:
plot = a.plot(y='roc_auc_score', marker='o', title='Mean ROC AUC score vs. Shared Percentage Threshold')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(a.index, a['roc_auc_score'], marker='o', label='a')
plt.plot(b.index, b['roc_auc_score'], marker='s', label='b')
plt.xlabel('th')
plt.ylabel('roc_auc_score')
plt.title('ROC AUC Score vs. Shared Percentage Threshold')
plt.legend()
plt.show()

In [ ]:
from CISM.cism.cism import GetMotifsResult

featureConf = HardDiscriminativeFC(extract_by=DiscriminativeFeatureKey.STRUCTURE_AND_CELL_IDENTITIES,
                                   use_cells_type_composition=False,
                                   use_motifs=True,
                                   shared_percentage=0.55,
                                   max_class_features=30,
                                   labels=classes)

motif_space_features = discriminator.get_features(feature_conf=featureConf,
                                                  exclude_patients=[],
                                                  n_jobs=1)

get_motifs_result = GetMotifsResult(motif_space_features=motif_space_features,
                                   discriminator=discriminator,
                                   classes=classes)

print(f'we have {len(get_motifs_result.all_motif_features)} discriminative motifs')

In [ ]:
dis_motifs = get_motifs_result.get_motifs_by_class()['CLR'] + get_motifs_result.get_motifs_by_class()['DII']

In [ ]:
t = discriminator.cism.motifs_dataset

In [ ]:
t

In [ ]:
wanted_cols = ['Patient','ID', 'Freq', 'Count','p_value', 'patient_class']
appearences = t[t['ID'].isin(dis_ids)][wanted_cols].sort_values(by=['ID', 'patient_class'])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

palette = sns.color_palette("Set2", len(dis_ids))

fig, axes = plt.subplots(len(dis_ids), 1, figsize=(8, 2 * len(dis_ids)), sharex=True)

for ax, id, color in zip(axes, dis_ids, palette):
    # Filter out rows with zero count
    motif_data = appearences[(appearences['ID'] == id) & (appearences['Count'] != 0)]
    
    bars = ax.bar(motif_data['Patient'], motif_data['Count'], color=color)
    ax.set_title(f"Motif ID: {id}")
    ax.set_xlabel("Patient")
    ax.set_ylabel("Count")
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.set_xticks(range(len(motif_data['Patient'])))
    ax.set_xticklabels(motif_data['Patient'], rotation=0)

    # Optional: add numeric value labels on top of bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, height + 0.05, f"{height:.0f}", 
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
wanted_cols = ['ID', 'Freq', 'Count','p_value', 'patient_class']

aggregated_motifs = t[t['ID'].isin(dis_motifs)][wanted_cols].groupby(['ID']).agg(
    {'Freq': 'mean', 'Count': 'mean', 'p_value': 'mean', 'patient_class': 'count'}
)

aggregated_motifs.columns = [col.strip() for col in aggregated_motifs.columns.to_flat_index()]
aggregated_motifs = aggregated_motifs.reset_index() 

aggregated_motifs

In [ ]:
aggregated_motifs.plot.scatter(x='patient_class', y='Count')

In [ ]:
aggregated_motifs.sort_values(by='Count', ascending=False).head(10)

In [ ]:
dis_ids = [12689602028,	538983436, 1616384140]

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt
from CISM.cism.helpers import create_weight_matrix_from_motifs

plt.figure(figsize=(20,8))

sns.set(font_scale=2)
sns.set_style("white")
sns.set_context("paper", font_scale=2.5)

df = create_weight_matrix_from_motifs(discriminator=discriminator,
                                      cells_type=cells_type,
                                      cell_identity_to_motif_hash=get_motifs_result.cell_identity_to_motif_hash,
                                      motifs_ids=get_motifs_result.get_motifs_by_class()[classes[0]],
                                      motifs_weight=get_motifs_result.get_motifs_mean_weight())

ax = sns.heatmap(df, annot=True, mask=df==0, cmap='Blues', annot_kws={'size': 14}, fmt=".3f")
ax.set_title(f'{classes[0]} motif-induced disease state')

In [ ]:

plt.figure(figsize=(20,8))

sns.set(font_scale=2)
sns.set_style("white")
sns.set_context("paper", font_scale=2.5)

df = create_weight_matrix_from_motifs(discriminator=discriminator,
                                      cells_type=cells_type,
                                      cell_identity_to_motif_hash=get_motifs_result.cell_identity_to_motif_hash,
                                      motifs_ids=get_motifs_result.get_motifs_by_class()[classes[1]],
                                      motifs_weight=get_motifs_result.get_motifs_mean_weight())

ax = sns.heatmap(df, annot=True, mask=df==0, cmap='Blues', annot_kws={'size': 14}, fmt=".3f")
ax.set_title(f'{classes[1]} motif-induced disease state')

In [ ]:
get_motifs_result.get_motifs_by_class()

In [ ]:
mots = get_motifs_result.get_motifs_mean_weight()

mots.sort_values(ascending=False) 

In [ ]:
from CISM.cism import helpers
from CISM.cism.vis import draw

for motif_id in dis_ids:
    hash_id = get_motifs_result.motif_to_cells_identity_hash[motif_id]
    print(f'motif_id: {motif_id}, hash_id: {hash_id}')
    target_motif = discriminator.cism.motifs_dataset[discriminator.cism.motifs_dataset.ID == motif_id].iloc[0].motif
    target_motif = helpers.string_base64_pickle(target_motif)
    draw.draw_motif(target_motif, cells_type=cells_type)

# Pairwise Analysis

In [ ]:
#read graphs
from CISM.pairwise.reader import GraphReader

raw_data_folder='CRC_graphs'
raw_data_folder_type=''

full_graph_df = pd.DataFrame()
full_graph_df = GraphReader.get_graphs(full_graph_df=full_graph_df,
                                       raw_data_folder='CRC_graphs',
                                       raw_data_folder_type='',
                                       disease='CRC',
                                       cells_type=cells_type)

In [ ]:
full_graph_df.graph.transform(lambda x: x.number_of_nodes()).describe()

In [ ]:
from CISM.pairwise.pairwise_model import PairwiseAnalysis


In [ ]:

tissue_state_to_string = {'CLR':'CLR', 'DII':'DII'}


pairwise_analysis = PairwiseAnalysis(
    tissue_state_csv_path='patient_class_simple.csv',
    tissue_state_to_string=tissue_state_to_string)

### fix columns for pairwise analysis

In [ ]:
from CISM.pairwise.pairwise_model import Columns

pairwise_analysis._patient_class_df[Columns.PATIENT_CLASS] = pairwise_analysis._patient_class_df[Columns.PATIENT_CLASS_ID]

In [ ]:
pairwise_analysis._patient_class_df[Columns.PATIENT_CLASS] = pairwise_analysis._patient_class_df[Columns.PATIENT_CLASS].apply(lambda row: "CLR" if row == 1 else "DII")
pairwise_analysis._patient_class_df['class'] = pairwise_analysis._patient_class_df[Columns.PATIENT_CLASS]

In [ ]:
cell_type_counts = pairwise_analysis.get_cell_type_count_from_classes(
                            full_graph_df=full_graph_df,
                            classes=['CLR', 'DII'],
                            cells_type=cells_type,
                            normalize=True)


In [ ]:
import numpy as np

roc_auc_scores = pairwise_analysis.analyze(full_graph_df=full_graph_df,
                                           cells_type=cells_type,
                                           labels=['CLR', 'DII'],
                                           trials=10)

print(f'roc_auc_score - mean: {np.mean(roc_auc_scores)} std: {np.std(roc_auc_scores)}')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
pairwise_data = {}
for patient_class in ['CLR', 'DII']:
    pairwise_data[patient_class] = pairwise_analysis.get_pairwise_counter(
        full_graph_df=full_graph_df,
        cells_type=cells_type,
        patient_class=patient_class
    )

vmin = min(df.min().min() for df in pairwise_data.values())
vmax = max(df.max().max() for df in pairwise_data.values())

for patient_class in ['CLR', 'DII']:
    heatmap_data = pairwise_data[patient_class]
    mask = np.tril(np.ones_like(heatmap_data, dtype=bool), k=-1)
    plt.figure(figsize=(16, 8))
    ax = sns.heatmap(heatmap_data, annot=True, mask=mask, cmap='Blues', annot_kws={'size': 12}, fmt=".3f", vmin=vmin, vmax=vmax)
    ax.set_title(f"Pairwise Analysis - {patient_class}")
    plt.show()

In [ ]:
cell_type_distribution = cell_type_counts.groupby('patient_class').apply(
    lambda df: df[['Cell Type', 'Frequency']].set_index('Cell Type').to_dict()['Frequency']
)


In [ ]:
cell_type_distribution

# Singletom Classifier

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

cell_types_list = list(cell_types.keys())
y = np.arange(len(cell_types_list))
height = 0.35

clr_freq = cell_type_counts[cell_type_counts['patient_class'] == 'CLR'].set_index('Cell Type').reindex(cell_types_list)['Frequency']
dii_freq = cell_type_counts[cell_type_counts['patient_class'] == 'DII'].set_index('Cell Type').reindex(cell_types_list)['Frequency']

plt.figure(figsize=(20, 10))
plt.barh(y - height/2, clr_freq, height, label='CLR', color=class_to_color['CLR'], alpha=0.7)
plt.barh(y + height/2, dii_freq, height, label='DII', color=class_to_color['DII'], alpha=0.7)

plt.ylabel('Cell Type')
plt.xlabel('Frequency')
plt.title('Cell Type Frequency by Patient Class')
plt.yticks(y, cell_types_list)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
wanterd_columns = ['patient_id', 'fov', 'class']
individual_cell_type_counts = (
    data[wanterd_columns]
    .groupby(['patient_id', 'fov', 'class'])
    .size()
    .reset_index(name='count')
    .pivot_table(index=['patient_id', 'fov'], columns='class', values='count', fill_value=0)
    .reset_index()
)

In [ ]:
counts_cols = [col for col in individual_cell_type_counts.columns if col not in ['patient_id', 'fov']]
individual_cell_type_counts['total_count'] = individual_cell_type_counts[counts_cols].sum(axis=1)
for col in counts_cols:
    individual_cell_type_counts[col] = individual_cell_type_counts[col] / individual_cell_type_counts['total_count']
individual_cell_type_counts.drop(columns='total_count', inplace=True)


In [ ]:
patient_class = pd.read_csv("patient_class_simple_shuffled.csv",header=None, names=['patient_id', 'class'])
patient_class['patient_id'] = patient_class['patient_id'].apply(lambda x: int(x[3:]))


In [ ]:
cell_frequesncies = individual_cell_type_counts.merge(patient_class, on='patient_id', how='left')
cell_frequesncies

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score
from random import shuffle
import random

accs = []
roc_aucs = []

for i in range(10):
    print(f'trial {i+1}/10')
    labels = list(patient_class['class'])
    random.seed(i)
    #shuffle(labels)
    patient_class['class'] = labels
    patient_class.to_csv(f'patient_class_simple_shuffled_{i}.csv', index=False, header=False)
    # Prepare features and labels
    X = cell_frequesncies.drop(columns=['patient_id', 'fov', 'class']).values
    y = cell_frequesncies['class'].values
    #shuffle(y)
    loo = LeaveOneOut()
    preds = []
    trues = []

    for train_idx, test_idx in loo.split(X):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        clf = RandomForestClassifier(random_state=random_state)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        preds.append(y_pred[0])
        trues.append(y_test[0])

    acc = accuracy_score(trues, preds)
    print(f'Leave-One-Out Random Forest Accuracy: {acc:.3f}')
    lb = LabelBinarizer()
    y_bin = lb.fit_transform(trues)

    preds_bin = lb.transform(preds)

    roc_auc = roc_auc_score(y_bin, preds_bin)
    print(f'Leave-One-Out Random Forest ROC AUC: {roc_auc:.3f}')
    accs.append(acc)
    roc_aucs.append(roc_auc)

print(f'Overall Accuracy: mean={np.mean(accs):.3f}, std={np.std(accs):.3f}')
print(f'Overall ROC AUC: mean={np.mean(roc_aucs):.3f}, std={np.std(roc_aucs):.3f}')

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelBinarizer

lb = LabelBinarizer()
y_bin = lb.fit_transform(trues)

preds_bin = lb.transform(preds)

roc_auc = roc_auc_score(y_bin, preds_bin)
print(f'Leave-One-Out Random Forest ROC AUC: {roc_auc:.3f}')


# Shuffle patient class labels

In [ ]:
patient_class = pd.read_csv('patient_class_simple.csv', header=None, names=['patient_id', 'class'])
labels = list(patient_class['class'])

In [ ]:
from random import shuffle
import random
random.seed(69420)
shuffle(labels)


In [ ]:
patient_class['class'] = labels

In [ ]:
patient_class.to_csv('patient_class_simple_shuffled.csv', index=False, header=False)

In [ ]:
trials = 10

for i in range(trials):
    print(f'trial {i+1}/{trials}')
    labels = list(patient_class['class'])
    random.seed(i)
    shuffle(labels)
    patient_class['class'] = labels
    patient_class.to_csv(f'patient_class_simple_shuffled_{i}.csv', index=False, header=False)
    

In [ ]:
def run_loocv(random_state, df, X_drop, y_col):
    #X  = df[X_cols].values
    X = df.drop(columns=['patient_id', 'fov', 'class']).values
    y = df['class'].values
    #shuffle(y)
    loo = LeaveOneOut()
    preds = []
    trues = []

    for train_idx, test_idx in loo.split(X):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        clf = RandomForestClassifier(random_state=random_state)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        preds.append(y_pred[0])
        trues.append(y_test[0])

    acc = accuracy_score(trues, preds)
    print(f'Leave-One-Out Random Forest Accuracy: {acc:.3f}')
    lb = LabelBinarizer()
    y_bin = lb.fit_transform(trues)

    preds_bin = lb.transform(preds)

    roc_auc = roc_auc_score(y_bin, preds_bin)
    print(f'Leave-One-Out Random Forest ROC AUC: {roc_auc:.3f}')